# Content-Based Filtering

### Data Loading

In [4]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import hstack

In [5]:
df = pd.read_csv('data/newMusicInfo.csv')
df.head()

,track_id,name,artist,spotify_preview_url,tags,genre,year,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,TROUVHL128F426C441,Come as You Are,NIRVANA,https://p.scdn.co/mp3-preview/a1c11bb1cb231031...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,218920,0.508,0.826,4,-5.783,0,0.0400,0.000175,0.000459,0.0878,0.543,120.012,4
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,https://p.scdn.co/mp3-preview/e7eb60e9466bc3a2...,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,238640,0.515,0.430,7,-9.935,1,0.0369,0.010200,0.000141,0.1290,0.104,91.841,4
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,https://p.scdn.co/mp3-preview/95cb9df1b056d759...,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,311014,0.566,0.429,5,-7.826,1,0.0242,0.715000,0.000014,0.1200,0.173,146.365,4
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,https://p.scdn.co/mp3-preview/90e41778392f27b6...,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,265506,0.554,0.490,4,-8.046,1,0.0457,0.016800,0.000534,0.1360,0.513,84.275,4
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,https://p.scdn.co/mp3-preview/7388c425022ced92...,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,223106,0.809,0.720,6,-7.776,1,0.1360,0.033700,0.001410,0.8260,0.832,138.556,4


### Selected only the relevant features

In [6]:
df_selected = df[['track_id', 'name','artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']]
df_selected.head()

,track_id,name,artist,tags,genre,year,danceability,speechiness,tempo
0,TROUVHL128F426C441,Come as You Are,NIRVANA,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, GRUNGE",RNB,1991,0.508,0.0400,120.012
1,TRLNZBD128F935E4D8,Creep,RADIOHEAD,"ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...",RNB,2008,0.515,0.0369,91.841
2,TRMZXEW128F9341FD5,The Scientist,COLDPLAY,"ROCK, ALTERNATIVE, INDIE, POP, ALTERNATIVE_ROC...",ROCK,2007,0.566,0.0242,146.365
3,TRIODZU128E078F3E2,Under the Bridge,RED HOT CHILI PEPPERS,"ROCK, ALTERNATIVE, ALTERNATIVE_ROCK, 90S, FUNK",POP,2003,0.554,0.0457,84.275
4,TRYEZGE128F92D04C8,Feel Good Inc.,GORILLAZ,"ROCK, ELECTRONIC, ALTERNATIVE, INDIE, POP, ALT...",RNB,2010,0.809,0.1360,138.556


### Check data type

In [7]:
df_selected.dtypes

track_id         object
name             object
artist           object
tags             object
genre            object
year              int64
danceability    float64
speechiness     float64
tempo           float64
dtype: object

### Copy and clean

In [8]:
df_selected = df[['track_id', 'name', 'artist', 'tags', 'genre', 'year', 'danceability', 'speechiness', 'tempo']].copy()
df_selected[['name', 'artist', 'tags', 'genre']] = df_selected[['name', 'artist', 'tags', 'genre']].fillna('')

### Combine text features

In [9]:
df_selected['combined_text'] = df_selected['tags'] + ' ' + df_selected['genre'] + ' ' + df_selected['artist']

### TF-IDF on combined text

In [10]:
# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 2),  # Include unigrams and bigrams
    min_df=1, # Keep words that appear in at least 1 document
    stop_words='english'
)

# Transform the combined text column into TF-IDF features
text_matrix = tfidf.fit_transform(df_selected['combined_text'])

### Normalize numeric features

In [11]:
# Initialize and apply MinMaxScaler to scale all numeric columns to [0, 1]
scaler = MinMaxScaler()
# Apply scaling to numerical features
numeric_matrix = scaler.fit_transform(df_selected[['danceability', 'speechiness', 'tempo', 'year']])

### Combine all features

In [12]:
combined_matrix = hstack([text_matrix, numeric_matrix]).tocsr()

### Fit KNN model

In [13]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(combined_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

### Build name and index mapping

In [14]:
# lowercase song names in mapping
# Build the (song name, year) -> index map
name_to_index = {
    (row['name'].lower().strip(), row['year']): idx
    for idx, row in df_selected.iterrows()
}

### Recommendation function using name

In [15]:
def recommend_songs_by_name_and_year(song_name, year, n_recommendations=5):
    song_key = (song_name.lower().strip(), year)
    
    if song_key not in name_to_index:
        return f"'{song_name}' from year {year} not found in the dataset."
    
    idx = name_to_index[song_key]
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # Exclude the input song
    
    input_song = df_selected.iloc[[idx]][['track_id', 'name', 'year', 'artist', 'genre', 'tags']]
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name', 'year', 'artist', 'genre', 'tags']]
    
    return pd.concat([input_song, recommendations]).reset_index(drop=True)


### Recommendation function using genre

In [16]:
def recommend_songs_by_genre_and_tempo(song_genre, song_tempo, n_recommendations=5):
    song_genre = song_genre.lower().strip()

    try:
        song_tempo = float(song_tempo)
    except ValueError:
        return "Tempo must be a number (BPM)."

    # Filter by genre
    genre_matches = df_selected[df_selected['genre'].str.lower() == song_genre]
    if genre_matches.empty:
        return f"Genre '{song_genre}' not found in the dataset."

    # Further filter by tempo (±5 BPM for flexibility)
    tempo_range = genre_matches[
        (genre_matches['tempo'] >= song_tempo - 5) & 
        (genre_matches['tempo'] <= song_tempo + 5)
    ]
    if tempo_range.empty:
        return f"No songs in genre '{song_genre}' within ±5 BPM of {song_tempo}."

    # Pick the first song from filtered matches to use in KNN
    idx = tempo_range.index[0]

    # Run KNN to find similar songs
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # Exclude the input song

    # Input song info
    input_song = df_selected.iloc[[idx]][['track_id', 'name', 'artist', 'genre', 'tempo']]

    # Filter recommended songs
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name', 'artist', 'genre', 'tempo']]
    recommendations = recommendations[~recommendations['track_id'].isin(input_song['track_id'])]

    return pd.concat([input_song, recommendations]).reset_index(drop=True)


### Recommendation function using tags

In [25]:
def recommend_songs_by_tags_and_danceability(song_tags, danceability_value, n_recommendations=5):
    song_tags = song_tags.lower().strip()
    
    try:
        danceability_value = float(danceability_value)
    except ValueError:
        return "Danceability must be a numeric value between 0.0 and 1.0."
    
    if not (0.0 <= danceability_value <= 1.0):
        return "Danceability must be between 0.0 and 1.0."

    # Filter by exact tags
    tag_matches = df_selected[df_selected['tags'].str.lower() == song_tags]
    if tag_matches.empty:
        return f"Tag '{song_tags}' not found in the dataset."

    # Further filter by danceability range (±0.1 for tolerance)
    filtered = tag_matches[
        (tag_matches['danceability'] >= danceability_value - 0.1) & 
        (tag_matches['danceability'] <= danceability_value + 0.1)
    ]
    
    if filtered.empty:
        return f"No songs with tag '{song_tags}' and danceability around {danceability_value}."

    # Use the closest match in danceability for similarity search
    idx = (filtered['danceability'] - danceability_value).abs().idxmin()


    # Run KNN
    distances, indices = knn_model.kneighbors(combined_matrix[idx], n_neighbors=n_recommendations + 1)
    recommended_indices = indices.flatten()[1:]  # exclude input

    # Input song
    input_song = df_selected.iloc[[idx]][['track_id', 'name', 'tags', 'danceability']]

    # Recommendations excluding already-selected song
    recommendations = df_selected.iloc[recommended_indices][['track_id', 'name','artist', 'tags', 'danceability']]
    recommendations = recommendations[~recommendations['track_id'].isin(input_song['track_id'])]

    return pd.concat([input_song, recommendations]).reset_index(drop=True)


###  User input

In [ ]:
song_input = input("Enter a song name: ").strip()
year_input = input("Enter the release year: ").strip()

try:
    year_input = int(year_input)
    recommendations = recommend_songs_by_name_and_year(song_input, year_input, n_recommendations=5)
    print("\n Recommended Songs:\n", recommendations)
except ValueError:
    print("Invalid year. Please enter a valid integer.")



 Recommended Songs:
              track_id          name  year     artist genre  \
0  TRLNZBD128F935E4D8         Creep  2008  RADIOHEAD   RNB   
1  TRIOUGL128F9310CA8        Lurgee  1993  RADIOHEAD  ROCK   
2  TRINQRE128F42BBAA2    Supersonic  2006      OASIS   RNB   
3  TRAFMBS128F423B54E   On Your Own  2000       BLUR   RNB   
4  TRSEHKD128E0782311   Coffee & TV  1999       BLUR  ROCK   
5  TRHSVNJ128E0780D7B  Pure Morning  2011    PLACEBO  ROCK   

                                                tags  
0  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
1  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
2  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
3  INDIE, ALTERNATIVE_ROCK, INDIE_ROCK, BRITISH, ...  
4  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  
5  ROCK, ALTERNATIVE, INDIE, ALTERNATIVE_ROCK, IN...  


In [ ]:
genre_input = input("Enter a genre separated by commas (e.g. Rock, electronic, Pop): ").strip()
tempo_input = input("Enter a tempo (BPM) (0-239): ").strip()

recommendations = recommend_songs_by_genre_and_tempo(genre_input, tempo_input, n_recommendations=5)
print("\n🎶 Recommended Songs:\n", recommendations)


🎶 Recommended Songs:
              track_id                         name    artist genre    tempo
0  TRMZXEW128F9341FD5                The Scientist  COLDPLAY  ROCK  146.365
1  TRAAHSY128F147BB5C               Speed of Sound  COLDPLAY  ROCK  123.085
2  TRUOJJS128F145C256  A Rush of Blood to the Head  COLDPLAY  ROCK  138.331
3  TRCIVWB128F92FDBB3             Strawberry Swing  COLDPLAY  ROCK  173.120
4  TRBGQHZ128F424EA56                    Bedshaped     KEANE  ROCK   74.065
5  TRIPSRK128F421A76D                     Daylight  COLDPLAY  ROCK  132.787


In [27]:
tag_input = input("Enter a tag (e.g. indie, folk, love): ").strip()
danceability_input = input("Enter danceability (0.0 to 1.0): ").strip()

recommendations = recommend_songs_by_tags_and_danceability(tag_input, danceability_input, n_recommendations=5)
print("\nRecommended Songs:\n", recommendations)



Recommended Songs:
              track_id                       name  \
0  TRIPRYQ128F4246915                   Starfire   
1  TRMNCGB128F4281AD5               Silver Rider   
2  TRELSNG128F4281ADE          Walk Into the Sea   
3  TROAFDA128F4281AD8            Cue the Strings   
4  TRPRAED128F429CBD4                   Disarray   
5  TREMFHA128F4281ADB  Broadway (So Many People)   

                            tags  danceability artist  
0                          INDIE         0.350    NaN  
1               INDIE, POST_ROCK         0.334    LOW  
2  ALTERNATIVE, INDIE, POST_ROCK         0.420    LOW  
3   INDIE, INDIE_ROCK, POST_ROCK         0.261    LOW  
4            POST_ROCK, SYNTHPOP         0.650    LOW  
5   INDIE, INDIE_ROCK, POST_ROCK         0.117    LOW  
